In [20]:
import yohou
import sys
import importlib
import yohou.base
import yohou.point_forecaster.base
import yohou.point_forecaster.reduction
importlib.reload(yohou.base)
importlib.reload(yohou.point_forecaster.base)
importlib.reload(yohou.point_forecaster.reduction)
from yohou.point_forecaster import PointReductionForecaster
print(yohou.__file__)
print(sys.path)

/home/gigi/Workspace/yohou/src/yohou/__init__.py
['/home/gigi/snap/code/215/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python313.zip', '/home/gigi/snap/code/215/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python3.13', '/home/gigi/snap/code/215/.local/share/uv/python/cpython-3.13.3-linux-x86_64-gnu/lib/python3.13/lib-dynload', '', '/home/gigi/Workspace/yohou/.venv/lib/python3.13/site-packages', '/home/gigi/Workspace/yohou/src']


In [13]:
from yohou.utils.polars import inspect_locality
print(inspect_locality(y))
print(inspect_locality(X))

(['a', 'b'], {})
(['c', 'd', 'e'], {})


In [21]:
import polars as pl
from datetime import datetime
from yohou.point_forecaster import PointReductionForecaster
from sklearn.linear_model import LinearRegression

length = 22
time = pl.datetime_range(
    start=datetime(2021, 12, 16),
    end=datetime(2021, 12, 16, 0, 0, length - 1),
    interval="1s",
    eager=True,
)
y = pl.DataFrame(
    {"time": time, "a": range(length), "b": range(10, length + 10)},
    schema={"time": pl.Datetime, "a": pl.Float64, "b": pl.Float64}
)
X = pl.DataFrame(
    {"time": time, "c": range(length), "d": range(10, length + 10), "e": range(20, length + 20)},
    schema={"time": pl.Datetime, "c": pl.Float64, "d": pl.Float64, "e": pl.Float64}
)

forecaster = PointReductionForecaster()
forecaster.fit(y=y, X=X, forecasting_horizon=1)
print("Fit done")
y_pred = forecaster.predict(forecasting_horizon=5)
print("Predict done")


DEBUG: _pre_fit X columns: ['time', 'c', 'd', 'e']
DEBUG: _pre_fit X_t columns: ['time', 'a', 'b', 'c', 'd', 'e']
Fit done


ColumnNotFoundError: unable to find column "c"; valid columns: ["a", "b"]

In [1]:
import polars as pl
import polars.selectors as cs
from datetime import datetime

In [2]:
observation_time = pl.datetime_range(
    start=datetime(2021, 12, 16),
    end=datetime(2021, 12, 16, 3),
    interval="1h",
    eager=True,
)

df = pl.DataFrame()
for time in observation_time:
    df_tmp = pl.DataFrame(
        {
            "time": [time],
            "global_a": [-1.0],
            "global_b": [1.0],
            "local_a": {
                "local_a_a": [
                    5.0,
                ],
                "local_a_b": [0.0],
            },
            "local_b": {"local_b_a": [5.0], "local_b_b": [0.0]},
        }
    )

    df = pl.concat([df, df_tmp])
print(df)

shape: (4, 5)
┌─────────────────────┬──────────┬──────────┬───────────┬───────────┐
│ time                ┆ global_a ┆ global_b ┆ local_a   ┆ local_b   │
│ ---                 ┆ ---      ┆ ---      ┆ ---       ┆ ---       │
│ datetime[μs]        ┆ f64      ┆ f64      ┆ struct[2] ┆ struct[2] │
╞═════════════════════╪══════════╪══════════╪═══════════╪═══════════╡
│ 2021-12-16 00:00:00 ┆ -1.0     ┆ 1.0      ┆ {5.0,0.0} ┆ {5.0,0.0} │
│ 2021-12-16 01:00:00 ┆ -1.0     ┆ 1.0      ┆ {5.0,0.0} ┆ {5.0,0.0} │
│ 2021-12-16 02:00:00 ┆ -1.0     ┆ 1.0      ┆ {5.0,0.0} ┆ {5.0,0.0} │
│ 2021-12-16 03:00:00 ┆ -1.0     ┆ 1.0      ┆ {5.0,0.0} ┆ {5.0,0.0} │
└─────────────────────┴──────────┴──────────┴───────────┴───────────┘


In [3]:
df.select(-pl.col("global_a").alias("a"))

a
f64
1.0
1.0
1.0
1.0


In [4]:
df.select(~cs.by_name("time"))

global_a,global_b,local_a,local_b
f64,f64,struct[2],struct[2]
-1.0,1.0,"{5.0,0.0}","{5.0,0.0}"
-1.0,1.0,"{5.0,0.0}","{5.0,0.0}"
-1.0,1.0,"{5.0,0.0}","{5.0,0.0}"
-1.0,1.0,"{5.0,0.0}","{5.0,0.0}"


In [5]:
pl.DataFrame({"1": df_tmp}).unnest("1")

time,global_a,global_b,local_a,local_b
datetime[μs],f64,f64,struct[2],struct[2]
2021-12-16 03:00:00,-1.0,1.0,"{5.0,0.0}","{5.0,0.0}"


In [6]:
df.select(["time", "local_a", "local_b", "global_b", "global_a"])

time,local_a,local_b,global_b,global_a
datetime[μs],struct[2],struct[2],f64,f64
2021-12-16 00:00:00,"{5.0,0.0}","{5.0,0.0}",1.0,-1.0
2021-12-16 01:00:00,"{5.0,0.0}","{5.0,0.0}",1.0,-1.0
2021-12-16 02:00:00,"{5.0,0.0}","{5.0,0.0}",1.0,-1.0
2021-12-16 03:00:00,"{5.0,0.0}","{5.0,0.0}",1.0,-1.0


In [7]:
df.schema["local_a"].fields[0].name

'local_a_a'

In [8]:
def check_continuity(df_p, df_n, expected_interval, check_intervals=True):
    time_p = df_p.select(cs.by_name("time"))[[-1]]
    time_n = df_n.select(cs.by_name("time"))[[0]]

    time = pl.concat([time_p, time_n])

    time_change = time.select(pl.col("time").diff()).fill_null(strategy="backward")
    interval = time_change[0, 0].seconds

    print(interval, expected_interval)
    if interval != expected_interval:
        raise ValueError()

In [9]:
%pdb 0
from yohou.utils.validation import check_interval_consistency

interval = check_interval_consistency(df)
interval

Automatic pdb calling has been turned OFF


/home/gigi/Workspace/yohou/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datetime.timedelta(seconds=3600)

In [10]:
check_continuity(df[:2], df[2:], interval.seconds)

3600 3600


In [11]:
observation_time = pl.datetime_range(
    start=datetime(2021, 12, 16),
    end=datetime(2021, 12, 16, 3),
    interval="1h",
    eager=True,
)

df_tmp = pl.DataFrame(
    {
        "time": observation_time,
        "g": range(len(observation_time)),
    }
)

In [12]:
df_tmp = df_tmp.join(df[["time"]], on="time").drop("time")

In [13]:
df_tmp.select(pl.all().abs().mean())

g
f64
1.5


In [14]:
(df_tmp - df_tmp).select(pl.all().abs().mean())

g
f64
0.0


In [15]:
df = df[
    [col for col, dtype in zip(df.columns, df.dtypes) if dtype != pl.Struct or col == "local_a"]
].unnest("local_a")

In [17]:
time_series_length = 22

df_ts = pl.DataFrame(
    {
        "time": pl.datetime_range(
            start=datetime(2021, 12, 16),
            end=datetime(2021, 12, 16, time_series_length - 1),
            interval="1h",
            eager=True,
        ),
        "a": range(time_series_length),
        "b": range(10, time_series_length + 10),
    }
)
df_ts

time,a,b
datetime[μs],i64,i64
2021-12-16 00:00:00,0,10
2021-12-16 01:00:00,1,11
2021-12-16 02:00:00,2,12
2021-12-16 03:00:00,3,13
2021-12-16 04:00:00,4,14
…,…,…
2021-12-16 17:00:00,17,27
2021-12-16 18:00:00,18,28
2021-12-16 19:00:00,19,29


In [18]:
time_series_length = 10
df_ts2 = pl.DataFrame(
    {
        "c": range(time_series_length),
        "d": range(10, time_series_length + 10),
    }
)

In [19]:
pl.concat([df_ts, df_ts2], how="horizontal")

time,a,b,c,d
datetime[μs],i64,i64,i64,i64
2021-12-16 00:00:00,0,10,0,10
2021-12-16 01:00:00,1,11,1,11
2021-12-16 02:00:00,2,12,2,12
2021-12-16 03:00:00,3,13,3,13
2021-12-16 04:00:00,4,14,4,14
…,…,…,…,…
2021-12-16 17:00:00,17,27,null,null
2021-12-16 18:00:00,18,28,null,null
2021-12-16 19:00:00,19,29,null,null


In [20]:
df_ts.select(pl.all())

time,a,b
datetime[μs],i64,i64
2021-12-16 00:00:00,0,10
2021-12-16 01:00:00,1,11
2021-12-16 02:00:00,2,12
2021-12-16 03:00:00,3,13
2021-12-16 04:00:00,4,14
…,…,…
2021-12-16 17:00:00,17,27
2021-12-16 18:00:00,18,28
2021-12-16 19:00:00,19,29


In [21]:
from sklearn.preprocessing import PowerTransformer

t = PowerTransformer().set_output(transform="polars")

t.fit_transform(df_ts)

time,a,b
f64,f64,f64
-1.655038,-1.888382,-1.731991
-1.497414,-1.623784,-1.548851
-1.33979,-1.394866,-1.369566
-1.182167,-1.18642,-1.193767
-1.024544,-0.991984,-1.021142
…,…,…
1.024543,1.001176,1.018693
1.182164,1.132899,1.163994
1.339785,1.262729,1.308023


In [22]:
from yohou.preprocessing import SeasonalDifferencing

t = SeasonalDifferencing(2)
X_t = t.fit_transform(df_ts)
df_ts

time,a,b
datetime[μs],i64,i64
2021-12-16 00:00:00,0,10
2021-12-16 01:00:00,1,11
2021-12-16 02:00:00,2,12
2021-12-16 03:00:00,3,13
2021-12-16 04:00:00,4,14
…,…,…
2021-12-16 17:00:00,17,27
2021-12-16 18:00:00,18,28
2021-12-16 19:00:00,19,29


In [23]:
t.inverse_transform(X_t, df_ts[:2])

time,a,b
datetime[μs],i64,i64
2021-12-16 02:00:00,2,12
2021-12-16 03:00:00,3,13
2021-12-16 04:00:00,4,14
2021-12-16 05:00:00,5,15
2021-12-16 06:00:00,6,16
…,…,…
2021-12-16 17:00:00,17,27
2021-12-16 18:00:00,18,28
2021-12-16 19:00:00,19,29


In [ ]:
def tabularize(df_time_series, window_length):
    df_tabular = (
        df_time_series.with_columns(
            [
                pl.col(col).shift(i).alias(f"{col}_lag_{i}")
                for (col, dtype) in zip(df_time_series.columns, df_time_series.dtypes)
                for i in range(window_length)
                if dtype != pl.Datetime
            ]
        )
    )[window_length - 1 :]

    return df_tabular

In [25]:
df_tabular = tabularize(df_ts, 3)
df_tabular

time,a,b,a_lag_0,a_lag_1,a_lag_2,b_lag_0,b_lag_1,b_lag_2
datetime[μs],i64,i64,i64,i64,i64,i64,i64,i64
2021-12-16 02:00:00,2,12,2,1,0,12,11,10
2021-12-16 03:00:00,3,13,3,2,1,13,12,11
2021-12-16 04:00:00,4,14,4,3,2,14,13,12
2021-12-16 05:00:00,5,15,5,4,3,15,14,13
2021-12-16 06:00:00,6,16,6,5,4,16,15,14
…,…,…,…,…,…,…,…,…
2021-12-16 17:00:00,17,27,17,16,15,27,26,25
2021-12-16 18:00:00,18,28,18,17,16,28,27,26
2021-12-16 19:00:00,19,29,19,18,17,29,28,27


In [26]:
df_tabular

time,a,b,a_lag_0,a_lag_1,a_lag_2,b_lag_0,b_lag_1,b_lag_2
datetime[μs],i64,i64,i64,i64,i64,i64,i64,i64
2021-12-16 02:00:00,2,12,2,1,0,12,11,10
2021-12-16 03:00:00,3,13,3,2,1,13,12,11
2021-12-16 04:00:00,4,14,4,3,2,14,13,12
2021-12-16 05:00:00,5,15,5,4,3,15,14,13
2021-12-16 06:00:00,6,16,6,5,4,16,15,14
…,…,…,…,…,…,…,…,…
2021-12-16 17:00:00,17,27,17,16,15,27,26,25
2021-12-16 18:00:00,18,28,18,17,16,28,27,26
2021-12-16 19:00:00,19,29,19,18,17,29,28,27


In [27]:
import numpy as np

a = np.array([1, 2, 3, 4, 5, 6])
a

array([1, 2, 3, 4, 5, 6])

In [28]:
a.reshape(2, 3).T

array([[1, 4],
       [2, 5],
       [3, 6]])

In [29]:
df_1 = pl.DataFrame(
    {
        "a": range(time_series_length),
    }
)

df_2 = pl.DataFrame(
    {
        "time": pl.datetime_range(
            start=datetime(2021, 12, 16),
            end=datetime(2021, 12, 16, time_series_length - 1),
            interval="1h",
            eager=True,
        ),
        "b": range(10, time_series_length + 10),
    }
)

In [30]:
fit_forecasting_horizon = 5
fit_interval = "1h"
seasonality = 3

In [31]:
df_2

time,b
datetime[μs],i64
2021-12-16 00:00:00,10
2021-12-16 01:00:00,11
2021-12-16 02:00:00,12
2021-12-16 03:00:00,13
2021-12-16 04:00:00,14
2021-12-16 05:00:00,15
2021-12-16 06:00:00,16
2021-12-16 07:00:00,17
2021-12-16 08:00:00,18


In [32]:
from datetime import datetime

df = pl.DataFrame(
    {
        "dt": [datetime(2022, 1, 1), datetime(2022, 1, 2)],
        "add": [1, 2],
    }
)
df
with pl.Config(tbl_width_chars=120):
    df.select(
        (pl.col("dt") + pl.duration(weeks="add")).alias("add_weeks"),
        (pl.col("dt") + pl.duration(days="add")).alias("add_days"),
        (pl.col("dt") + pl.duration(seconds="add")).alias("add_seconds"),
        (pl.col("dt") + pl.duration(milliseconds="add")).alias("add_millis"),
        (pl.col("dt") + pl.duration(hours="add")).alias("add_hours"),
    )

In [33]:
y = pl.DataFrame(
    {
        "time": pl.datetime_range(
            start=datetime(2021, 12, 16),
            end=datetime(2021, 12, 16, 0, 0, 0, 1),
            interval="54ns",
            eager=True,
        ),
    }
)
y

time
datetime[ns]
2021-12-16 00:00:00
2021-12-16 00:00:00.000000054
2021-12-16 00:00:00.000000108
2021-12-16 00:00:00.000000162
2021-12-16 00:00:00.000000216
…
2021-12-16 00:00:00.000000756
2021-12-16 00:00:00.000000810
2021-12-16 00:00:00.000000864


In [34]:
time_change = y.select(cs.by_name("time").diff()).fill_null(strategy="backward")

interval = time_change[0, 0]
assert len(time_change) == len(time_change.select(pl.col("time") == interval))

In [35]:
y.select(time=pl.col("time").dt.offset_by(f"{5 * interval[0, 0].seconds}s"))

TypeError: 'datetime.timedelta' object is not subscriptable

In [ ]:
df = pl.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6], "c": [7, 8, 9]})

cols_to_mean = ["a", "c"]

In [ ]:
df.select(pl.col(cols_to_mean))

a,c
i64,i64
1,7
2,8
3,9


In [ ]:
df.select(col_mean=pl.concat_list(cols_to_mean).list.mean())

col_mean
f64
4.0
5.0
6.0
